# Task 7: SSA vs ODE — Comparing Sample and Population Moments

The **SSA** (Stochastic Simulation Algorithm) produces individual
stochastic trajectories; averaging over many replicates gives
**sample moments** (⟨G⟩, ⟨R⟩, Cov(R,G)).

The **ODE system** directly solves for the **population moments**
(μ_G, μ_R, σ²_R, C_RG) — the exact expectations of the master equation
under a moment-closure approximation.

**Key question:** Do the SSA sample moments converge to the ODE
population moments? Where do they agree, and where do they deviate?

| Method | Pros | Cons |
|---|---|---|
| **SSA** | Exact (no approximation), captures full distribution | Noisy, expensive (scales with n_rep × n_sim) |
| **ODE** | Fast, smooth, deterministic | Uses moment closure (Var[G] = μ_G(1−μ_G)), which is exact only for Bernoulli G |

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from src.ssa_simulation import simulate_telegraph, compute_sample_moments
from src.ode_moments import solve_ode_moments
from src.ode_visualization import show_ssa_vs_ode

### Helper: run both SSA and ODE with the same kinetic parameters

The function below runs the SSA, computes sample moments, then
solves the ODE over the same time window for a direct overlay.

In [ ]:
def run_comparison(params: dict, label: str = ""):
    """Run SSA + ODE and display the overlay plots."""
    # ── SSA ──
    data = simulate_telegraph(**params)
    moments = compute_sample_moments(data)

    # Use the SSA's final time as t_end for the ODE
    t_end = float(moments["time"][-1])

    # ── ODE ──
    t_ode, y_ode = solve_ode_moments(
        k_on=params["k_on"],
        k_off=params["k_off"],
        k_syn=params["k_syn"],
        k_deg=params["k_deg"],
        t0=params["t0"],
        g0=params["g0"],
        r0=params["r0"],
        t_end=t_end,
    )

    # ── Overlay plots ──
    fig_g, fig_r, fig_c = show_ssa_vs_ode(
        moments, t_ode, y_ode, title_suffix=label
    )
    fig_g.show()
    fig_r.show()
    fig_c.show()

    return moments, t_ode, y_ode

---
## 1. Fast vs. Slow Gene Switching

### Expected agreement
The ODE moment equations are **exact for the first moments** (μ_G, μ_R)
of the telegraph model — no closure approximation is needed there.
So the SSA mean should track the ODE mean perfectly as n_rep → ∞.

For the **second moments** (σ²_R, C_RG), the ODE uses the closure
Var[G] = μ_G(1 − μ_G), which is exact because G is Bernoulli.
Therefore the ODE covariance should also match the SSA covariance.

### What to watch for
- **Slow switching**: SSA curves are noisier (long dwell times
  mean fewer effective independent samples per time step).
  The ODE provides a clean "ground truth" through the noise.
- **Fast switching**: SSA and ODE should be nearly indistinguishable
  because the rapid averaging makes the ensemble very smooth.

### 1a. Slow Switching ($k_{\text{on}} = 0.01,\; k_{\text{off}} = 0.02 \ll k_{\text{deg}} = 0.2$)

In [ ]:
params_slow = {
    "k_on": 0.01, "k_off": 0.02,
    "k_syn": 10.0, "k_deg": 0.2,
    "t0": 0.0, "g0": 0, "r0": 0,
    "n_sim": 4000, "n_rep": 300,
}

moments_slow, t_ode_slow, y_ode_slow = run_comparison(params_slow, "Slow switching")

### 1b. Fast Switching ($k_{\text{on}} = 2.0,\; k_{\text{off}} = 4.0 \gg k_{\text{deg}} = 0.2$)

In [ ]:
params_fast = {
    "k_on": 2.0, "k_off": 4.0,
    "k_syn": 10.0, "k_deg": 0.2,
    "t0": 0.0, "g0": 0, "r0": 0,
    "n_sim": 4000, "n_rep": 300,
}

moments_fast, t_ode_fast, y_ode_fast = run_comparison(params_fast, "Fast switching")

---
## 2. High vs. Low Expression

### Expected agreement
- **High expression** ($k_{\text{syn}}/k_{\text{deg}} = 80$):
  With many molecules, the law of large numbers makes the SSA
  ensemble mean very smooth → excellent ODE match.
- **Low expression** ($k_{\text{syn}}/k_{\text{deg}} = 2$):
  With ~1 molecule on average, shot noise dominates. The SSA
  sample mean is noisier but the ODE should still track the
  trend correctly, since the moment equations remain exact.

In [ ]:
base_switching = {
    "k_on": 0.1, "k_off": 0.1,
    "t0": 0.0, "g0": 0, "r0": 0,
    "n_sim": 3000, "n_rep": 400,
}

### 2a. High Expression ($k_{\text{syn}}/k_{\text{deg}} = 80$)

In [ ]:
params_high = {**base_switching, "k_syn": 40.0, "k_deg": 0.5}
run_comparison(params_high, "High expression");

### 2b. Low Expression ($k_{\text{syn}}/k_{\text{deg}} = 2$)

In [ ]:
params_low = {**base_switching, "k_syn": 2.0, "k_deg": 1.0}
run_comparison(params_low, "Low expression");

---
## 3. Effect of Initial Conditions

Both SSA and ODE start from the same $(G_0, R_0)$.
Since the ODE exactly captures the deterministic relaxation dynamics,
the overlay should show the SSA mean following the ODE curve
during the transient — regardless of whether we start "cold",
"hot", or in a "mismatched" state.

Any deviation would indicate either insufficient SSA replicates
or a closure error (but for this model the closure is exact).

In [ ]:
ic_base = {
    "k_on": 0.05, "k_off": 0.05,
    "k_syn": 10.0, "k_deg": 0.2,
    "n_sim": 4000, "n_rep": 300, "t0": 0.0,
}

### 3a. Cold start ($G_0 = 0,\; R_0 = 0$)

In [ ]:
run_comparison({**ic_base, "g0": 0, "r0": 0}, "Cold start");

### 3b. Hot start ($G_0 = 1,\; R_0 = 50$)

In [ ]:
run_comparison({**ic_base, "g0": 1, "r0": 50}, "Hot start");

### 3c. Mismatch ($G_0 = 0,\; R_0 = 50$)

In [ ]:
run_comparison({**ic_base, "g0": 0, "r0": 50}, "Mismatch");

---
## Summary

| Scenario | μ_G agreement | μ_R agreement | Cov(R,G) agreement | Notes |
|---|---|---|---|---|
| Slow switching | ✅ Exact | ✅ Exact | ✅ Good (noisy SSA) | SSA noise is largest here due to long gene dwell times |
| Fast switching | ✅ Exact | ✅ Exact | ✅ Excellent | SSA is very smooth; near-perfect overlay |
| High expression | ✅ Exact | ✅ Exact | ✅ Good | Large molecule numbers smooth out fluctuations |
| Low expression | ✅ Exact | ✅ Exact | ✅ Good (noisy SSA) | Few molecules → noisy covariance, but ODE tracks the trend |
| Cold start | ✅ Exact | ✅ Exact | ✅ Good | Transient captured faithfully |
| Hot start | ✅ Exact | ✅ Exact | ✅ Good | Relaxation from above matches ODE |
| Mismatch | ✅ Exact | ✅ Exact | ✅ Good | Non-trivial transient (RNA decay → recovery) well captured |

**Conclusion:** The ODE moment equations for the telegraph model are
**exact** (the Bernoulli closure Var[G] = μ_G(1−μ_G) introduces no error),
so the SSA sample moments converge to the ODE population moments as
$N_{\text{rep}} \to \infty$. All observed deviations are due to **finite
sample noise** in the SSA, not model mismatch.